# 풍투데이 크롤링 (풍투데이 차트 기준)

## 1. 버튜버 '릴파' 의 2026년 04월 누적 별풍선, 시급(풍) 테스트

In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import re

# 크롬 실행
driver = webdriver.Chrome()

# 페이지 열기
driver.get("https://poong.today/broadcast/lilpa0309")

# 로딩 기다리기
time.sleep(3)

# 전체 텍스트 가져오기
body_text = driver.find_element(By.TAG_NAME, "body").text

# 데이터 추출
total_poong = re.search(r"누적\s*별풍선\s*([\d,]+)", body_text)
hourly_poong = re.search(r"시급\s*\(풍\)\s*([\d,]+)", body_text)

# 출력
if total_poong:
    print("누적 별풍선:", total_poong.group(1))
else:
    print("누적 별풍선 못찾음")

if hourly_poong:
    print("시급(풍):", hourly_poong.group(1))
else:
    print("시급 못찾음")

# 종료
driver.quit()

누적 별풍선: 116,556
시급(풍): 3,363


## 2. 버튜버 '릴파' 의 2025.01~2026.03 누적 별풍선, 시급(풍) CSV

In [37]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import pandas as pd

driver = webdriver.Chrome()
driver.get("https://poong.today/broadcast/lilpa0309")

time.sleep(5)

results = []

# 현재 2026-04부터 시작 → 2025-01까지 (총 15개월)
for i in range(16):

    time.sleep(2)

    # li 기반으로 데이터 추출
    lis = driver.find_elements(By.TAG_NAME, "li")

    total_star = None
    hourly_star = None

    for li in lis:
        text = li.text

        if "누적 별풍선" in text:
            total_star = li.find_element(By.TAG_NAME, "span").text

        if "시급(풍)" in text:
            hourly_star = li.find_element(By.TAG_NAME, "span").text

    # 숫자 변환 추가
    total_star = int(total_star.replace(",", "")) if total_star else None
    hourly_star = int(hourly_star.replace(",", "")) if hourly_star else None
    
    # 년 / 월 추출
    year = None
    month = None

    try:
        year = int(driver.find_element(By.CLASS_NAME, "small-year").text.strip())
        month = int(driver.find_element(By.CLASS_NAME, "small-month").text.strip())
    except:
        year, month = None, None
    if not (year == 2026 and month == 4):
        results.append({
            "year": year,
            "month": month,
            "total_star": total_star,
            "hourly_star": hourly_star
        })

    print(f"{year}-{month} / {total_star} / {hourly_star}")

    # 이전 달 버튼 클릭
    try:
        prev_btn = driver.find_element(By.CSS_SELECTOR, ".small-month .left-arrow")
        prev_btn.click()
    except Exception as e:
        print("버튼 실패:", e)
        break

driver.quit()

df = pd.DataFrame(results)
df.to_csv("lilpa_monthly_final.csv", index=False, encoding="utf-8-sig")

print("CSV 저장 완료!")

2026-4 / 116556 / 3363
2026-3 / 813222 / 6847
2026-2 / 375888 / 4903
2026-1 / 263410 / 2398
2025-12 / 356332 / 4373
2025-11 / 315947 / 4662
2025-10 / 223786 / 5942
2025-9 / 182842 / 1776
2025-8 / 298683 / 5500
2025-7 / 356011 / 5789
2025-6 / 189101 / 1779
2025-5 / 524130 / 9674
2025-4 / 272795 / 4266
2025-3 / 692774 / 6936
2025-2 / 483872 / 4839
2025-1 / 448074 / 7984
CSV 저장 완료!


In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   year         14 non-null     int64
 1   month        14 non-null     int64
 2   total_star   14 non-null     int64
 3   hourly_star  14 non-null     int64
dtypes: int64(4)
memory usage: 580.0 bytes


## 3. 풍투데이 별풍선 랭킹 TOP 100 버튜버의 2025.01~2026.03 누적 별풍선, 시급(풍) CSV 

### 버츄얼 TOP 100  스트리머 기간별 누적/시급 풍 수집 (지난달기준)

#### TOP 100 채널ID 리스트 만들기

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# 브라우저 설정
options = Options()
options.add_argument("start-maximized")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# 사이트 접속
driver.get("https://poong.today/rankings/broadcast")

# 1. 카테고리 드롭다운 열기
dropdown = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, ".react-dropdown-select"))
)
driver.execute_script("arguments[0].click();", dropdown)

# 2. 버추얼 선택
virtual_option = wait.until(
    EC.presence_of_element_located((By.XPATH, "//span[@role='option' and contains(text(),'버추얼')]"))
)
driver.execute_script("arguments[0].click();", virtual_option)

time.sleep(2)

# 3. 지난 달 클릭
last_month = wait.until(
    EC.presence_of_element_located((By.XPATH, "//a[contains(text(),'지난 달')]"))
)
driver.execute_script("arguments[0].click();", last_month)

time.sleep(3)

# 4. 더보기 눌러서 100위까지 확장
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(2)

more_btn = wait.until(
    EC.presence_of_element_located((By.XPATH, "//button[contains(text(),'더보기')]"))
)
driver.execute_script("arguments[0].click();", more_btn)

time.sleep(3)

# 5. li.row 기준으로 데이터 추출
rows = driver.find_elements(By.CSS_SELECTOR, "li.row")

data = []

for row in rows:
    try:
        rank = row.find_element(By.CSS_SELECTOR, ".rank").text
        name = row.find_element(By.CSS_SELECTOR, ".nick").text.split("\n")[0]
        
        link = row.find_element(By.CSS_SELECTOR, ".subject a").get_attribute("href")
        channel_id = link.split("/")[-1]
        
        data.append([rank, name, channel_id])
        
    except:
        continue

df = pd.DataFrame(data, columns=["rank", "name", "channel_id"])

print(df.head())
print("총 개수:", len(df))

driver.quit()

  rank  name   channel_id
0    1   릴파♬    lilpa0309
1    2    누야   nuyaonline
2    3  하루아이   loveya4860
3    4  연초록♪  whatcherry4
4    5  킹냥이_       sa5791
총 개수: 100


In [59]:
channel_id_list = df['channel_id'].tolist()
print(channel_id_list)
print(len(channel_id_list))

['lilpa0309', 'nuyaonline', 'loveya4860', 'whatcherry4', 'sa5791', 'singgyul', 'likey0u', '243000', 'gosegu2', 'chaenna02', 'navixx', 'deathhammer', 'zuu1028', 'gurm01', 'sjsr4611', 'tleod1818', 'wlssud44', 'psb010203', 'breezy25', 'bboringirl', 'jeongwazzang', 'kaksjak0730', 'plincess', 'youxhh', 'duwoor', 'dnwnwjdqhr53', 'qooqoo3110', 'yeveee', 'sl0724', 'highfeel3391', 'vf3366', 'duddj4210', 'kmj1tkfkd1go', 'cctvno', 'nanamoon777', 'lighthoong', 'hinqocorv', 'kcw1217', 'saehorshu', 'kyaang123', 'jingburger1', 'rapja2025', 'yuna812', 'chiyo75', 'kgoyangyeeee', 'aengduwoo', 'naturexx1', 'toree0409', 'hika12', 'ozan201', 'darklucy', 'hani0320', 'toocat030', 'soondeoki', 'choiagain', 'qw794', 'kplooto122', 'goyamii', 'insome0319', 'simkoong0102', 'ldrboo', 'viichan6', 'madaomm', 'baeksoon2', 'beemong', 'rintube', '0zilimi0', 'xxxx922', 'bbungchi', 'cotton1217', 'carrot99', 'donggeul2', 'imganzino', 'riabelle0325', 'amaiyk0105', 'pinktape8', 'yjkim5500', 'os3n0o', 'pumong', 'rmlrl771', '

#### 수집

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

options = Options()
options.add_argument("start-maximized")
driver = webdriver.Chrome(options=options)

all_results = [] # 100명의 데이터를 모두 담길 리스트

# 2. 100명의 BJ ID를 순회
for channel_id in channel_id_list:
    print(f"[{channel_id}] 데이터 수집 시작...")
    
    url = f"https://poong.today/broadcast/{channel_id}"
    driver.get(url)
    
    time.sleep(4) 
    
    for i in range(16): 
        time.sleep(1.5) 
        
        try:
            # li 기반으로 데이터 추출
            lis = driver.find_elements(By.TAG_NAME, "li")
            total_star, hourly_star = None, None
            
            for li in lis:
                text = li.text
                if "누적 별풍선" in text:
                    total_star = li.find_element(By.TAG_NAME, "span").text
                if "시급(풍)" in text:
                    hourly_star = li.find_element(By.TAG_NAME, "span").text
            
            # 숫자 변환
            total_star = int(total_star.replace(",", "")) if total_star else None
            hourly_star = int(hourly_star.replace(",", "")) if hourly_star else None
            
            # 년 / 월 추출
            year = int(driver.find_element(By.CLASS_NAME, "small-year").text.strip())
            month = int(driver.find_element(By.CLASS_NAME, "small-month").text.strip())
            
            # 2026년 4월 데이터 제외
            if not (year == 2026 and month == 4):
                all_results.append({
                    "channel_id": channel_id, 
                    "year": year,
                    "month": month,
                    "total_star": total_star,
                    "hourly_star": hourly_star
                })
                print(f"{year}-{month:02d} / 누적: {total_star} / 시급: {hourly_star}")
            
            # 이전 달 버튼 클릭
            prev_btn = driver.find_element(By.CSS_SELECTOR, ".small-month .left-arrow")
            prev_btn.click()
            
        except Exception as e:
            # 방송을 시작한 지 15개월이 안 된 신입 BJ이거나 예기치 못한 구조 변경 시
            print(f"[{channel_id}] 과거 데이터가 더 없거나 확인 불가. 다음 BJ로 넘어갑니다.")
            break
            
    print("-" * 50)

driver.quit()

# 4. 최종 데이터프레임 저장
df_final = pd.DataFrame(all_results)
df_final.to_csv("top100_vtuber_poong.csv", index=False, encoding="utf-8-sig")

print(f"총 {len(df_final)}행의 데이터가 CSV로 저장되었습니다!")

🚀 [lilpa0309] 데이터 수집 시작...
   ㄴ 2026-03 / 누적: 813222 / 시급: 6847
   ㄴ 2026-02 / 누적: 375888 / 시급: 4903
   ㄴ 2026-01 / 누적: 263410 / 시급: 2398
   ㄴ 2025-12 / 누적: 356332 / 시급: 4373
   ㄴ 2025-11 / 누적: 315947 / 시급: 4662
   ㄴ 2025-10 / 누적: 223786 / 시급: 5942
   ㄴ 2025-09 / 누적: 182842 / 시급: 1776
   ㄴ 2025-08 / 누적: 298683 / 시급: 5500
   ㄴ 2025-07 / 누적: 356011 / 시급: 5789
   ㄴ 2025-06 / 누적: 189101 / 시급: 1779
   ㄴ 2025-05 / 누적: 524130 / 시급: 9674
   ㄴ 2025-04 / 누적: 272795 / 시급: 4266
   ㄴ 2025-03 / 누적: 692774 / 시급: 6936
   ㄴ 2025-02 / 누적: 483872 / 시급: 4839
   ㄴ 2025-01 / 누적: 448074 / 시급: 7984
--------------------------------------------------
🚀 [nuyaonline] 데이터 수집 시작...
   ㄴ 2026-03 / 누적: 736938 / 시급: 8046
   ㄴ 2026-02 / 누적: 1466890 / 시급: 17605
   ㄴ 2026-01 / 누적: 1161474 / 시급: 12317
   ㄴ 2025-12 / 누적: 187979 / 시급: 2192
   ㄴ 2025-11 / 누적: 177670 / 시급: 1897
   ㄴ 2025-10 / 누적: 293191 / 시급: 4091
   ㄴ 2025-09 / 누적: 171547 / 시급: 2054
   ㄴ 2025-08 / 누적: 182754 / 시급: 2041
   ㄴ 2025-07 / 누적: 213451 / 시급: 1806
  

#### name도 붙인 버전

In [ ]:
import pandas as pd

df_data = pd.read_csv("top100_vtuber_poong.csv")

df_merged = pd.merge(df_data, df, on='channel_id', how='left')

# 컬럼 순서 재배치
df_merged = df_merged[['rank', 'name', 'channel_id', 'year', 'month', 'total_star', 'hourly_star']]

df_merged.to_csv("top100_vtuber_poong_last_month.csv", index=False, encoding="utf-8-sig")

print("병합 완료")
print(df_merged.head(10))

병합 완료
  rank name channel_id  year  month  total_star  hourly_star
0    1  릴파♬  lilpa0309  2026      3      813222         6847
1    1  릴파♬  lilpa0309  2026      2      375888         4903
2    1  릴파♬  lilpa0309  2026      1      263410         2398
3    1  릴파♬  lilpa0309  2025     12      356332         4373
4    1  릴파♬  lilpa0309  2025     11      315947         4662
5    1  릴파♬  lilpa0309  2025     10      223786         5942
6    1  릴파♬  lilpa0309  2025      9      182842         1776
7    1  릴파♬  lilpa0309  2025      8      298683         5500
8    1  릴파♬  lilpa0309  2025      7      356011         5789
9    1  릴파♬  lilpa0309  2025      6      189101         1779


### 버츄얼 TOP 101~103  스트리머 기간별 누적/시급 풍 수집 테스트

In [76]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# 1. 브라우저 설정
options = Options()
options.add_argument("start-maximized")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# 2. 사이트 접속 및 필터 설정
driver.get("https://poong.today/rankings/broadcast")

# 카테고리 드롭다운 열기
dropdown = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".react-dropdown-select")))
driver.execute_script("arguments[0].click();", dropdown)

# 버추얼 선택
virtual_option = wait.until(EC.presence_of_element_located((By.XPATH, "//span[@role='option' and contains(text(),'버추얼')]")))
driver.execute_script("arguments[0].click();", virtual_option)
time.sleep(2)

# 지난 달 클릭
last_month = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(text(),'지난 달')]")))
driver.execute_script("arguments[0].click();", last_month)
time.sleep(3)

# 3. 목표 순위(103위)가 보일 때까지 '더보기' 버튼 반복 클릭
target_max_rank = 103

while True:
    # 현재 로드된 전체 목록 수 확인
    rows = driver.find_elements(By.CSS_SELECTOR, "li.row")
    
    # 충분히 로드되었으면 더보기 중단
    if len(rows) >= target_max_rank:
        print(f"{len(rows)}명의 데이터가 로드되었습니다. 추출을 시작합니다.")
        break

    # 스크롤을 맨 아래로 내려서 더보기 버튼이 잘 보이게 함
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)
    
    try:
        more_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//button[contains(text(),'더보기')]")))
        driver.execute_script("arguments[0].click();", more_btn)
        time.sleep(2) # 로딩 대기
    except:
        print("🚨 더 이상 '더보기' 버튼을 찾을 수 없습니다. (마지막 페이지 도달)")
        break

# 4. 101위 ~ 103위 데이터만 필터링하여 추출
rows = driver.find_elements(By.CSS_SELECTOR, "li.row")
data = []

for row in rows:
    try:
        rank_text = row.find_element(By.CSS_SELECTOR, ".rank").text
        if not rank_text.isdigit():
            continue
            
        rank = int(rank_text)
        
        # 101위부터 103위까지만 수집
        if 101 <= rank <= 103:
            name = row.find_element(By.CSS_SELECTOR, ".nick").text.split("\n")[0]
            link = row.find_element(By.CSS_SELECTOR, ".subject a").get_attribute("href")
            channel_id = link.split("/")[-1]
            
            data.append([rank, name, channel_id])
            print(f"수집 완료: {rank}위 - {name} ({channel_id})")
            
        # 103위를 넘어가면 불필요한 반복문 종료
        elif rank > 103:
            break
            
    except Exception as e:
        continue

# 5. 데이터프레임 변환 및 확인
df_test = pd.DataFrame(data, columns=["rank", "name", "channel_id"])
print("\n[수집 결과 확인]")
print(df_test)

driver.quit()

150명의 데이터가 로드되었습니다. 추출을 시작합니다.
수집 완료: 101위 - 하나링구 (skfk6612)
수집 완료: 102위 - 따스히 (guqlswls123)
수집 완료: 103위 - 빛나봄 (nabombom)

[수집 결과 확인]
   rank  name   channel_id
0   101  하나링구     skfk6612
1   102   따스히  guqlswls123
2   103   빛나봄     nabombom


In [77]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

channel_id_list = df_test['channel_id'].tolist() 

options = Options()
options.add_argument("start-maximized")
driver = webdriver.Chrome(options=options)

all_results = []

for channel_id in channel_id_list:
    print(f"▶ [{channel_id}] 상세 데이터 수집 시작...")
    url = f"https://poong.today/broadcast/{channel_id}"
    driver.get(url)
    time.sleep(4) 
    

    for i in range(16): 
        time.sleep(1.5) 
        try:
            # 연/월 추출
            year = int(driver.find_element(By.CLASS_NAME, "small-year").text.strip())
            month = int(driver.find_element(By.CLASS_NAME, "small-month").text.strip())
            
            # 2026년 4월 데이터 제외
            if not (year == 2026 and month == 4):
                # 데이터 추출
                lis = driver.find_elements(By.TAG_NAME, "li")
                total_star, hourly_star = None, None
                
                for li in lis:
                    text = li.text
                    if "누적 별풍선" in text:
                        total_star = li.find_element(By.TAG_NAME, "span").text
                    if "시급(풍)" in text:
                        hourly_star = li.find_element(By.TAG_NAME, "span").text
                
                # 숫자 변환
                total_star = int(total_star.replace(",", "")) if total_star else 0
                hourly_star = int(hourly_star.replace(",", "")) if hourly_star else 0
                
                all_results.append({
                    "channel_id": channel_id, 
                    "year": year,
                    "month": month,
                    "total_star": total_star,
                    "hourly_star": hourly_star
                })
                print(f"   - {year}-{month:02d} 수집: {total_star:,}풍 / 시급 {hourly_star:,}풍")
            
            # 이전 달
            prev_btn = driver.find_element(By.CSS_SELECTOR, ".small-month .left-arrow")
            prev_btn.click()
            
        except Exception as e:
            print(f"   - {channel_id}: 더 이상의 과거 기록이 없습니다.")
            break
            
    print("-" * 30)

driver.quit()

# 데이터 병합
df_detail = pd.DataFrame(all_results)
# 랭킹 정보(df_test)와 상세 정보(df_detail) 병합
df_final = pd.merge(df_test, df_detail, on='channel_id', how='left')

df_final = df_final[['rank', 'name', 'channel_id', 'year', 'month', 'total_star', 'hourly_star']]

print("\n[최종 병합 데이터 결과]")
print(df_final.head(15))

# 테스트용 파일 저장
df_final.to_csv("test_101_103_vtuber.csv", index=False, encoding="utf-8-sig")
print("총",len(df_final),'개 저장완료했습니다.')

▶ [skfk6612] 상세 데이터 수집 시작...
   - 2026-03 수집: 115,386풍 / 시급 1,241풍
   - 2026-02 수집: 191,210풍 / 시급 2,367풍
   - 2026-01 수집: 119,792풍 / 시급 1,338풍
   - 2025-12 수집: 169,761풍 / 시급 1,273풍
   - 2025-11 수집: 171,775풍 / 시급 1,004풍
   - 2025-10 수집: 258,471풍 / 시급 1,253풍
   - 2025-09 수집: 136,745풍 / 시급 971풍
   - 2025-08 수집: 5,242풍 / 시급 677풍
   - 2025-07 수집: 4,933풍 / 시급 781풍
   - 2025-06 수집: 60,865풍 / 시급 949풍
   - 2025-05 수집: 102,413풍 / 시급 754풍
   - 2025-04 수집: 200,246풍 / 시급 1,010풍
   - 2025-03 수집: 39,510풍 / 시급 518풍
   - 2025-02 수집: 120,742풍 / 시급 1,462풍
   - 2025-01 수집: 280,170풍 / 시급 1,276풍
------------------------------
▶ [guqlswls123] 상세 데이터 수집 시작...
   - 2026-03 수집: 115,012풍 / 시급 211풍
   - 2026-02 수집: 142,499풍 / 시급 303풍
   - 2026-01 수집: 286,858풍 / 시급 754풍
   - 2025-12 수집: 108,514풍 / 시급 235풍
   - 2025-11 수집: 140,355풍 / 시급 255풍
   - 2025-10 수집: 101,182풍 / 시급 237풍
   - 2025-09 수집: 68,636풍 / 시급 381풍
   - 2025-08 수집: 262,761풍 / 시급 512풍
   - 2025-07 수집: 161,259풍 / 시급 336풍
   - 2025-06 수집: 90,430풍 / 시급 299

## 테스트 완료! 

#### 구간별로 나눠서 수집

### 버츄얼 TOP 101~400 스트리머 기간별 누적/시급 풍 수집 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# 1. 브라우저 설정
options = Options()
options.add_argument("start-maximized")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# 2. 사이트 접속 및 필터 설정
driver.get("https://poong.today/rankings/broadcast")

# 카테고리 드롭다운 열기
dropdown = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".react-dropdown-select")))
driver.execute_script("arguments[0].click();", dropdown)

# 버추얼 선택
virtual_option = wait.until(EC.presence_of_element_located((By.XPATH, "//span[@role='option' and contains(text(),'버추얼')]")))
driver.execute_script("arguments[0].click();", virtual_option)
time.sleep(2)

# 지난 달 클릭
last_month = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(text(),'지난 달')]")))
driver.execute_script("arguments[0].click();", last_month)
time.sleep(3)

# 3. 목표 순위(103위)가 보일 때까지 '더보기' 버튼 반복 클릭
target_max_rank = 400

while True:
    # 현재 로드된 전체 목록 수 확인
    rows = driver.find_elements(By.CSS_SELECTOR, "li.row")
    
    # 충분히 로드되었으면 더보기 중단
    if len(rows) >= target_max_rank:
        print(f"{len(rows)}명의 데이터가 로드되었습니다. 추출을 시작합니다.")
        break

    # 스크롤을 맨 아래로 내려서 더보기 버튼이 잘 보이게 함
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)
    
    try:
        more_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//button[contains(text(),'더보기')]")))
        driver.execute_script("arguments[0].click();", more_btn)
        time.sleep(2) 
    except:
        print("더 이상 '더보기' 버튼을 찾을 수 없습니다. (마지막 페이지 도달)")
        break

# 4. 101위 ~ 400위 데이터만 필터링하여 추출
rows = driver.find_elements(By.CSS_SELECTOR, "li.row")
data = []

for row in rows:
    try:
        rank_text = row.find_element(By.CSS_SELECTOR, ".rank").text
        if not rank_text.isdigit():
            continue
            
        rank = int(rank_text)
        
        # 101위부터 400위까지만 수집
        if 101 <= rank <= 400:
            name = row.find_element(By.CSS_SELECTOR, ".nick").text.split("\n")[0]
            link = row.find_element(By.CSS_SELECTOR, ".subject a").get_attribute("href")
            channel_id = link.split("/")[-1]
            
            data.append([rank, name, channel_id])
            print(f"수집 완료: {rank}위 - {name} ({channel_id})")
            
        # 400위를 넘어가면 불필요한 반복문 종료
        elif rank > 400:
            break
            
    except Exception as e:
        continue

# 5. 데이터프레임 변환 및 확인
df_test = pd.DataFrame(data, columns=["rank", "name", "channel_id"])
print("\n[수집 결과 확인]")
print(df_test)

driver.quit()

400명의 데이터가 로드되었습니다. 추출을 시작합니다.
수집 완료: 101위 - 하나링구 (skfk6612)
수집 완료: 102위 - 따스히 (guqlswls123)
수집 완료: 103위 - 빛나봄 (nabombom)
수집 완료: 104위 - 냥쏘 (leesoi34)
수집 완료: 105위 - 둥찌 (doongzzi)
수집 완료: 106위 - 세르 (yourseru)
수집 완료: 107위 - 루나♩ (ltg8022)
수집 완료: 108위 - 라벤_ (lavend0710)
수집 완료: 109위 - 체비_ (chebi2)
수집 완료: 110위 - 소심해_ (thtlago0607)
수집 완료: 111위 - 히뚜* (rud21458)
수집 완료: 112위 - _임재천 (t730172)
수집 완료: 113위 - _한시아 (sia030)
수집 완료: 114위 - 마로니! (marronie)
수집 완료: 115위 - 라무 (5959700)
수집 완료: 116위 - 부르 (bureu2002)
수집 완료: 117위 - 포로미_ (sim222)
수집 완료: 118위 - 멜로딩딩 (melodingding)
수집 완료: 119위 - 비숑b (merryou)
수집 완료: 120위 - 키르__ (qhelrkem28)
수집 완료: 121위 - 미주. (omizuxoxo)
수집 완료: 122위 - 한세긴 (rose0957)
수집 완료: 123위 - 큐피 (jjuppi1022)
수집 완료: 124위 - 엘리시아 (282eyo)
수집 완료: 125위 - 캬루♥ (kyaru96)
수집 완료: 126위 - 데네브. (denebeu)
수집 완료: 127위 - 달체솜 (dalchesom)
수집 완료: 128위 - 에이레네 (eirene0326)
수집 완료: 129위 - 여르미_ (yeorumi030)
수집 완료: 130위 - 츄르CHUR (chur1004)
수집 완료: 131위 - 라히ㅀ (doraheee)
수집 완료: 132위 - 해솔♡ (liloolil)
수집 완료: 133위 - 펭퀸♡ (peng

In [75]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

channel_id_list = df_test['channel_id'].tolist() 

options = Options()
options.add_argument("start-maximized")
driver = webdriver.Chrome(options=options)

all_results = []

for channel_id in channel_id_list:
    print(f"[{channel_id}] 상세 데이터 수집 시작...")
    url = f"https://poong.today/broadcast/{channel_id}"
    driver.get(url)
    time.sleep(4) 
    

    for i in range(16): 
        time.sleep(1.5) 
        try:
            # 연/월 추출
            year = int(driver.find_element(By.CLASS_NAME, "small-year").text.strip())
            month = int(driver.find_element(By.CLASS_NAME, "small-month").text.strip())
            
            # 2026년 4월 데이터 제외
            if not (year == 2026 and month == 4):
                # 데이터 추출
                lis = driver.find_elements(By.TAG_NAME, "li")
                total_star, hourly_star = None, None
                
                for li in lis:
                    text = li.text
                    if "누적 별풍선" in text:
                        total_star = li.find_element(By.TAG_NAME, "span").text
                    if "시급(풍)" in text:
                        hourly_star = li.find_element(By.TAG_NAME, "span").text
                
                # 숫자 변환
                total_star = int(total_star.replace(",", "")) if total_star else 0
                hourly_star = int(hourly_star.replace(",", "")) if hourly_star else 0
                
                all_results.append({
                    "channel_id": channel_id, 
                    "year": year,
                    "month": month,
                    "total_star": total_star,
                    "hourly_star": hourly_star
                })
                print(f"   - {year}-{month:02d} 수집: {total_star:,}풍 / 시급 {hourly_star:,}풍")
            
            # 이전 달
            prev_btn = driver.find_element(By.CSS_SELECTOR, ".small-month .left-arrow")
            prev_btn.click()
            
        except Exception as e:
            print(f"   - {channel_id}: 더 이상의 과거 기록이 없습니다.")
            break
            
    print("-" * 30)

driver.quit()

# 데이터 병합
df_detail = pd.DataFrame(all_results)
# 랭킹 정보(df_test)와 상세 정보(df_detail) 병합
df_final = pd.merge(df_test, df_detail, on='channel_id', how='left')

df_final = df_final[['rank', 'name', 'channel_id', 'year', 'month', 'total_star', 'hourly_star']]

print("\n[최종 병합 데이터 결과]")
print(df_final.head(15))

# 테스트용 파일 저장
df_final.to_csv("test_101_400_vtuber.csv", index=False, encoding="utf-8-sig")

[skfk6612] 상세 데이터 수집 시작...
   - 2026-03 수집: 115,386풍 / 시급 1,241풍
   - 2026-02 수집: 191,210풍 / 시급 2,367풍
   - 2026-01 수집: 119,792풍 / 시급 1,338풍
   - 2025-12 수집: 169,761풍 / 시급 1,273풍
   - 2025-11 수집: 171,775풍 / 시급 1,004풍
   - 2025-10 수집: 258,471풍 / 시급 1,253풍
   - 2025-09 수집: 136,745풍 / 시급 971풍
   - 2025-08 수집: 5,242풍 / 시급 677풍
   - 2025-07 수집: 4,933풍 / 시급 781풍
   - 2025-06 수집: 60,865풍 / 시급 949풍
   - 2025-05 수집: 102,413풍 / 시급 754풍
   - 2025-04 수집: 200,246풍 / 시급 1,010풍
   - 2025-03 수집: 39,510풍 / 시급 518풍
   - 2025-02 수집: 120,742풍 / 시급 1,462풍
   - 2025-01 수집: 280,170풍 / 시급 1,276풍
------------------------------
[guqlswls123] 상세 데이터 수집 시작...
   - 2026-03 수집: 115,012풍 / 시급 211풍
   - 2026-02 수집: 142,499풍 / 시급 303풍
   - 2026-01 수집: 286,858풍 / 시급 754풍
   - 2025-12 수집: 108,514풍 / 시급 235풍
   - 2025-11 수집: 140,355풍 / 시급 255풍
   - 2025-10 수집: 101,182풍 / 시급 237풍
   - 2025-09 수집: 68,636풍 / 시급 381풍
   - 2025-08 수집: 262,761풍 / 시급 512풍
   - 2025-07 수집: 161,259풍 / 시급 336풍
   - 2025-06 수집: 90,430풍 / 시급 299풍
  

### 버츄얼 TOP 401~700 스트리머 기간별 누적/시급 풍 수집 

In [85]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# 1. 브라우저 설정
options = Options()
options.add_argument("start-maximized")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# 2. 사이트 접속 및 필터 설정
driver.get("https://poong.today/rankings/broadcast")

# 카테고리 드롭다운 열기
dropdown = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".react-dropdown-select")))
driver.execute_script("arguments[0].click();", dropdown)

# 버추얼 선택
virtual_option = wait.until(EC.presence_of_element_located((By.XPATH, "//span[@role='option' and contains(text(),'버추얼')]")))
driver.execute_script("arguments[0].click();", virtual_option)
time.sleep(2)

# 지난 달 클릭
last_month = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(text(),'지난 달')]")))
driver.execute_script("arguments[0].click();", last_month)
time.sleep(3)

# 3. 목표 순위(103위)가 보일 때까지 '더보기' 버튼 반복 클릭
target_max_rank = 700

while True:
    # 현재 로드된 전체 목록 수 확인
    rows = driver.find_elements(By.CSS_SELECTOR, "li.row")
    
    # 충분히 로드되었으면 더보기 중단
    if len(rows) >= target_max_rank:
        print(f"{len(rows)}명의 데이터가 로드되었습니다. 추출을 시작합니다.")
        break

    # 스크롤을 맨 아래로 내려서 더보기 버튼이 잘 보이게 함
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)
    
    try:
        more_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//button[contains(text(),'더보기')]")))
        driver.execute_script("arguments[0].click();", more_btn)
        time.sleep(2) 
    except:
        print("더 이상 '더보기' 버튼을 찾을 수 없습니다. (마지막 페이지 도달)")
        break

# 4. 101위 ~ 400위 데이터만 필터링하여 추출
rows = driver.find_elements(By.CSS_SELECTOR, "li.row")
data = []

for row in rows:
    try:
        rank_text = row.find_element(By.CSS_SELECTOR, ".rank").text
        if not rank_text.isdigit():
            continue
            
        rank = int(rank_text)
        
        # 101위부터 400위까지만 수집
        if 401 <= rank <= 700:
            name = row.find_element(By.CSS_SELECTOR, ".nick").text.split("\n")[0]
            link = row.find_element(By.CSS_SELECTOR, ".subject a").get_attribute("href")
            channel_id = link.split("/")[-1]
            
            data.append([rank, name, channel_id])
            print(f"수집 완료: {rank}위 - {name} ({channel_id})")
            
        # 400위를 넘어가면 불필요한 반복문 종료
        elif rank > 700:
            break
            
    except Exception as e:
        continue

# 5. 데이터프레임 변환 및 확인
df_test = pd.DataFrame(data, columns=["rank", "name", "channel_id"])
print("\n[수집 결과 확인]")
print(df_test)

driver.quit()

700명의 데이터가 로드되었습니다. 추출을 시작합니다.
수집 완료: 401위 - 백시온_ (a685rwzkfsq)
수집 완료: 402위 - 딴딴2당 (dbsk0708)
수집 완료: 403위 - 키요_ (kiyozz11)
수집 완료: 404위 - 시냐비 (sebyeok)
수집 완료: 405위 - 갈푸짱 (juin0123)
수집 완료: 406위 - 구월이_ (isq1158)
수집 완료: 407위 - 키젤_ (rydwl0214)
수집 완료: 408위 - 방우리_ (lsh8071)
수집 완료: 409위 - 윤이결 (khaea218)
수집 완료: 410위 - 코코양 (cococ11)
수집 완료: 411위 - 김박하◇ (bakha0804)
수집 완료: 412위 - 메루♪ (meuuu3)
수집 완료: 413위 - 목츄리 (xex3)
수집 완료: 414위 - 히무루 (himuru)
수집 완료: 415위 - 크앙희 (9wang2)
수집 완료: 416위 - 온담 (gangj486)
수집 완료: 417위 - 하르. (hareu99)
수집 완료: 418위 - 김막댕 (kimmakdaeng)
수집 완료: 419위 - 다뮤_ (not15987)
수집 완료: 420위 - 감자가비 (doki0818)
수집 완료: 421위 - 다니얀 (daniyan1030)
수집 완료: 422위 - 유시엘 (yusiela)
수집 완료: 423위 - 진비유 (mahiruchans2)
수집 완료: 424위 - 유세람 (eunseo5o)
수집 완료: 425위 - 연자몽 (ojingeo11)
수집 완료: 426위 - 뢴트게늄 (jey422)
수집 완료: 427위 - 산동@ (kimscello5)
수집 완료: 428위 - 나모모짱 (dduddumoon)
수집 완료: 429위 - 표표표3 (pyo32o)
수집 완료: 430위 - 시루냥 (xirus2)
수집 완료: 431위 - 란다_ (top6373)
수집 완료: 432위 - 라일락_lilac (lilacev)
수집 완료: 433위 - 어쩜냥이 (hhr001234)


In [86]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

channel_id_list = df_test['channel_id'].tolist() 

options = Options()
options.add_argument("start-maximized")
driver = webdriver.Chrome(options=options)

all_results = []

for channel_id in channel_id_list:
    print(f"[{channel_id}] 상세 데이터 수집 시작...")
    url = f"https://poong.today/broadcast/{channel_id}"
    driver.get(url)
    time.sleep(4) 
    

    for i in range(16): 
        time.sleep(1.5) 
        try:
            # 연/월 추출
            year = int(driver.find_element(By.CLASS_NAME, "small-year").text.strip())
            month = int(driver.find_element(By.CLASS_NAME, "small-month").text.strip())
            
            # 2026년 4월 데이터 제외
            if not (year == 2026 and month == 4):
                # 데이터 추출
                lis = driver.find_elements(By.TAG_NAME, "li")
                total_star, hourly_star = None, None
                
                for li in lis:
                    text = li.text
                    if "누적 별풍선" in text:
                        total_star = li.find_element(By.TAG_NAME, "span").text
                    if "시급(풍)" in text:
                        hourly_star = li.find_element(By.TAG_NAME, "span").text
                
                # 숫자 변환
                total_star = int(total_star.replace(",", "")) if total_star else 0
                hourly_star = int(hourly_star.replace(",", "")) if hourly_star else 0
                
                all_results.append({
                    "channel_id": channel_id, 
                    "year": year,
                    "month": month,
                    "total_star": total_star,
                    "hourly_star": hourly_star
                })
                print(f"   - {year}-{month:02d} 수집: {total_star:,}풍 / 시급 {hourly_star:,}풍")
            
            # 이전 달
            prev_btn = driver.find_element(By.CSS_SELECTOR, ".small-month .left-arrow")
            prev_btn.click()
            
        except Exception as e:
            print(f"   - {channel_id}: 더 이상의 과거 기록이 없습니다.")
            break
            
    print("-" * 30)

driver.quit()

# 데이터 병합
df_detail = pd.DataFrame(all_results)
# 랭킹 정보(df_test)와 상세 정보(df_detail) 병합
df_final = pd.merge(df_test, df_detail, on='channel_id', how='left')

df_final = df_final[['rank', 'name', 'channel_id', 'year', 'month', 'total_star', 'hourly_star']]

print("\n[최종 병합 데이터 결과]")
print(df_final.head(15))

# 테스트용 파일 저장
df_final.to_csv("test_401_700_vtuber.csv", index=False, encoding="utf-8-sig")
print('완료했습니다')
print("총",len(df_final),'개 저장완료했습니다.')

[a685rwzkfsq] 상세 데이터 수집 시작...
   - 2026-03 수집: 35,616풍 / 시급 348풍
   - 2026-02 수집: 45,476풍 / 시급 390풍
   - 2026-01 수집: 38,683풍 / 시급 391풍
   - 2025-12 수집: 14,569풍 / 시급 331풍
   - 2025-11 수집: 8,524풍 / 시급 210풍
   - 2025-10 수집: 6,645풍 / 시급 201풍
   - 2025-09 수집: 18,275풍 / 시급 350풍
   - 2025-08 수집: 63,822풍 / 시급 798풍
   - 2025-07 수집: 39,540풍 / 시급 397풍
   - 2025-06 수집: 63,709풍 / 시급 536풍
   - 2025-05 수집: 63,134풍 / 시급 704풍
   - 2025-04 수집: 18,791풍 / 시급 331풍
   - 2025-03 수집: 16,446풍 / 시급 344풍
   - 2025-02 수집: 8,622풍 / 시급 239풍
   - 2025-01 수집: 0풍 / 시급 0풍
------------------------------
[dbsk0708] 상세 데이터 수집 시작...
   - 2026-03 수집: 35,483풍 / 시급 131풍
   - 2026-02 수집: 58,865풍 / 시급 232풍
   - 2026-01 수집: 83,011풍 / 시급 485풍
   - 2025-12 수집: 67,582풍 / 시급 254풍
   - 2025-11 수집: 73,018풍 / 시급 282풍
   - 2025-10 수집: 144,119풍 / 시급 479풍
   - 2025-09 수집: 102,463풍 / 시급 329풍
   - 2025-08 수집: 71,290풍 / 시급 245풍
   - 2025-07 수집: 73,292풍 / 시급 243풍
   - 2025-06 수집: 36,413풍 / 시급 162풍
   - 2025-05 수집: 35,122풍 / 시급 169풍
   - 2025-

### 버츄얼 TOP 701~945 스트리머 기간별 누적/시급 풍 수집 

In [82]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# 1. 브라우저 설정
options = Options()
options.add_argument("start-maximized")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# 2. 사이트 접속 및 필터 설정
driver.get("https://poong.today/rankings/broadcast")

# 카테고리 드롭다운 열기
dropdown = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".react-dropdown-select")))
driver.execute_script("arguments[0].click();", dropdown)

# 버추얼 선택
virtual_option = wait.until(EC.presence_of_element_located((By.XPATH, "//span[@role='option' and contains(text(),'버추얼')]")))
driver.execute_script("arguments[0].click();", virtual_option)
time.sleep(2)

# 지난 달 클릭
last_month = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(text(),'지난 달')]")))
driver.execute_script("arguments[0].click();", last_month)
time.sleep(3)

# 3. 목표 순위(103위)가 보일 때까지 '더보기' 버튼 반복 클릭
target_max_rank = 945

while True:
    # 현재 로드된 전체 목록 수 확인
    rows = driver.find_elements(By.CSS_SELECTOR, "li.row")
    
    # 충분히 로드되었으면 더보기 중단
    if len(rows) >= target_max_rank:
        print(f"{len(rows)}명의 데이터가 로드되었습니다. 추출을 시작합니다.")
        break

    # 스크롤을 맨 아래로 내려서 더보기 버튼이 잘 보이게 함
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)
    
    try:
        more_btn = wait.until(EC.presence_of_element_located((By.XPATH, "//button[contains(text(),'더보기')]")))
        driver.execute_script("arguments[0].click();", more_btn)
        time.sleep(2) 
    except:
        print("더 이상 '더보기' 버튼을 찾을 수 없습니다. (마지막 페이지 도달)")
        break

# 4. 101위 ~ 400위 데이터만 필터링하여 추출
rows = driver.find_elements(By.CSS_SELECTOR, "li.row")
data = []

for row in rows:
    try:
        rank_text = row.find_element(By.CSS_SELECTOR, ".rank").text
        if not rank_text.isdigit():
            continue
            
        rank = int(rank_text)
        
        # 101위부터 400위까지만 수집
        if 701 <= rank <= 945:
            name = row.find_element(By.CSS_SELECTOR, ".nick").text.split("\n")[0]
            link = row.find_element(By.CSS_SELECTOR, ".subject a").get_attribute("href")
            channel_id = link.split("/")[-1]
            
            data.append([rank, name, channel_id])
            print(f"수집 완료: {rank}위 - {name} ({channel_id})")
            
        # 400위를 넘어가면 불필요한 반복문 종료
        elif rank > 945:
            break
            
    except Exception as e:
        continue

# 5. 데이터프레임 변환 및 확인
df_test = pd.DataFrame(data, columns=["rank", "name", "channel_id"])
print("\n[수집 결과 확인]")
print(df_test)

driver.quit()

945명의 데이터가 로드되었습니다. 추출을 시작합니다.
수집 완료: 701위 - 비상구9 (swa07037)
수집 완료: 702위 - 피너 (sinoost2)
수집 완료: 703위 - 뚱땅★ (o8o8o8o8o)
수집 완료: 704위 - ㆍ포포ㆍ (sunza1122)
수집 완료: 705위 - 슈타- (rockshooter)
수집 완료: 706위 - 승돌♬ (nhyebeenpark)
수집 완료: 707위 - 해시 (nhaesi17)
수집 완료: 708위 - 허니론 (seochol)
수집 완료: 709위 - 마리히 (marihema)
수집 완료: 710위 - 공주샤샤 (syasya0w0)
수집 완료: 711위 - 죠아써 (jjoasseo13)
수집 완료: 712위 - ㅋl스 (ehdhks0552)
수집 완료: 713위 - 유이슬 (0108916)
수집 완료: 714위 - 하티하티 (gkxl1004)
수집 완료: 715위 - 하눙씌 (hanoong)
수집 완료: 716위 - 강삐닝 (biin0728)
수집 완료: 717위 - 데로DERO (b4bycthu1hu)
수집 완료: 718위 - 햄이! (hamchichi)
수집 완료: 719위 - 미미짱짱세용 (n1574cat)
수집 완료: 720위 - ♡치야미 (chiyami)
수집 완료: 721위 - 롤리__ (rolli826)
수집 완료: 722위 - 김유지! (nhiyuji0710)
수집 완료: 723위 - 빵땅콩 (dmng50)
수집 완료: 724위 - ♥히메 (momoseh1me)
수집 완료: 725위 - 달세뇨_ (fkgpf007)
수집 완료: 726위 - 개보린_ (whoru1127)
수집 완료: 727위 - 깡수지니 (asj0430)
수집 완료: 728위 - 아유앙레아엔 (ayu0918v)
수집 완료: 729위 - 진은해 (dasiruu)
수집 완료: 730위 - 쿠로미츠치리 (chirik0)
수집 완료: 731위 - 제노현생 (gen010)
수집 완료: 732위 - 밤문_ (gome02)
수집 완료: 73

In [83]:
print(df_test)

     rank     name   channel_id
0     701     비상구9     swa07037
1     702       피너     sinoost2
2     703      뚱땅★    o8o8o8o8o
3     704     ㆍ포포ㆍ    sunza1122
4     705      슈타-  rockshooter
..    ...      ...          ...
240   941      유주,     yuju0120
241   942     엄석대_     kwoo0221
242   943      메리*   meryvtuber
243   944     슈슈땅♥  knachthexen
244   945  버추얼SOOP    afvirtual

[245 rows x 3 columns]


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

channel_id_list = df_test['channel_id'].tolist() 

options = Options()
options.add_argument("start-maximized")
driver = webdriver.Chrome(options=options)

all_results = []

for channel_id in channel_id_list:
    print(f"[{channel_id}] 상세 데이터 수집 시작...")
    url = f"https://poong.today/broadcast/{channel_id}"
    driver.get(url)
    time.sleep(4) 
    

    for i in range(16): 
        time.sleep(1.5) 
        try:
            # 연/월 추출
            year = int(driver.find_element(By.CLASS_NAME, "small-year").text.strip())
            month = int(driver.find_element(By.CLASS_NAME, "small-month").text.strip())
            
            # 2026년 4월 데이터 제외
            if not (year == 2026 and month == 4):
                # 데이터 추출
                lis = driver.find_elements(By.TAG_NAME, "li")
                total_star, hourly_star = None, None
                
                for li in lis:
                    text = li.text
                    if "누적 별풍선" in text:
                        total_star = li.find_element(By.TAG_NAME, "span").text
                    if "시급(풍)" in text:
                        hourly_star = li.find_element(By.TAG_NAME, "span").text
                
                # 숫자 변환
                total_star = int(total_star.replace(",", "")) if total_star else 0
                hourly_star = int(hourly_star.replace(",", "")) if hourly_star else 0
                
                all_results.append({
                    "channel_id": channel_id, 
                    "year": year,
                    "month": month,
                    "total_star": total_star,
                    "hourly_star": hourly_star
                })
                print(f"   - {year}-{month:02d} 수집: {total_star:,}풍 / 시급 {hourly_star:,}풍")
            
            # 이전 달
            prev_btn = driver.find_element(By.CSS_SELECTOR, ".small-month .left-arrow")
            prev_btn.click()
            
        except Exception as e:
            print(f"   - {channel_id}: 더 이상의 과거 기록이 없습니다.")
            break
            
    print("-" * 30)

driver.quit()

# 데이터 병합
df_detail = pd.DataFrame(all_results)
# 랭킹 정보(df_test)와 상세 정보(df_detail) 병합
df_final = pd.merge(df_test, df_detail, on='channel_id', how='left')

df_final = df_final[['rank', 'name', 'channel_id', 'year', 'month', 'total_star', 'hourly_star']]

print("\n[최종 병합 데이터 결과]")
print(df_final.head(15))

# 테스트용 파일 저장
df_final.to_csv("test_701_945_vtuber.csv", index=False, encoding="utf-8-sig")
print('완료했습니다')
print("총",len(df_final),'개 저장완료했습니다.')

[swa07037] 상세 데이터 수집 시작...
   - 2026-03 수집: 6,561풍 / 시급 175풍
   - 2026-02 수집: 4,523풍 / 시급 91풍
   - 2026-01 수집: 3,104풍 / 시급 55풍
   - 2025-12 수집: 3,071풍 / 시급 46풍
   - 2025-11 수집: 585풍 / 시급 8풍
   - 2025-10 수집: 3,380풍 / 시급 47풍
   - 2025-09 수집: 1,285풍 / 시급 17풍
   - 2025-08 수집: 4,484풍 / 시급 58풍
   - 2025-07 수집: 3,243풍 / 시급 43풍
   - 2025-06 수집: 0풍 / 시급 0풍
   - 2025-05 수집: 0풍 / 시급 0풍
   - 2025-04 수집: 0풍 / 시급 0풍
   - 2025-03 수집: 0풍 / 시급 0풍
   - 2025-02 수집: 0풍 / 시급 0풍
   - 2025-01 수집: 0풍 / 시급 0풍
------------------------------
[sinoost2] 상세 데이터 수집 시작...
   - 2026-03 수집: 6,502풍 / 시급 195풍
   - 2026-02 수집: 5,529풍 / 시급 85풍
   - 2026-01 수집: 31,641풍 / 시급 316풍
   - 2025-12 수집: 1,742풍 / 시급 50풍
   - 2025-11 수집: 16,968풍 / 시급 80풍
   - 2025-10 수집: 9,840풍 / 시급 79풍
   - 2025-09 수집: 14,592풍 / 시급 144풍
   - 2025-08 수집: 4,055풍 / 시급 46풍
   - 2025-07 수집: 2,315풍 / 시급 33풍
   - 2025-06 수집: 296풍 / 시급 12풍
   - 2025-05 수집: 11,273풍 / 시급 70풍
   - 2025-04 수집: 16,010풍 / 시급 68풍
   - 2025-03 수집: 3,980풍 / 시급 27풍
   - 2025-02 수집: 

# 구간 모두 합치기

In [97]:
import pandas as pd

# 합칠 파일 리스트
files = [
    'top100_vtuber_poong_last_month.csv',
    'test_101_400_vtuber.csv',
    'test_401_700_vtuber.csv',
    'test_701_945_vtuber.csv'
]

# 모든 파일을 읽어서 리스트에 담기
df_list = []
for file in files:
    try:
        df = pd.read_csv(file)
        df_list.append(df)
        print(f"'{file}' 읽기 완료: {len(df)}행")
    except FileNotFoundError:
        print(f"오류: '{file}' 파일을 찾을 수 없습니다.")

# 데이터 합치기 (세로 방향)
if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # 결과 저장
    output_filename = 'poongtoday_vtuber_rankings.csv'
    combined_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n성공적으로 합쳐졌습니다! 총 {len(combined_df)}개의 데이터가 '{output_filename}'에 저장되었습니다.")

'top100_vtuber_poong_last_month.csv' 읽기 완료: 1500행
'test_101_400_vtuber.csv' 읽기 완료: 4500행
'test_401_700_vtuber.csv' 읽기 완료: 4500행
'test_701_945_vtuber.csv' 읽기 완료: 3647행

성공적으로 합쳐졌습니다! 총 14147개의 데이터가 'poongtoday_vtuber_rankings.csv'에 저장되었습니다.
